In [ ]:
import requests
import json
import time

auth = json.loads(mssparkutils.notebook.run("procore_auth"))
token = auth["token"]
COMPANY_ID = auth["company_id"]
headers = {
    "Authorization": f"Bearer {token}",
    "Procore-Company-Id": str(COMPANY_ID)
}

print("Auth successful" if token else "Auth failed")

StatementMeta(, b0f5b176-13c5-4283-b56f-2bd6dc11b560, 3, Finished, Available, Finished, False)

Auth successful


In [2]:
projects_response = requests.get(
    "https://api.procore.com/rest/v1.0/projects",
    headers=headers,
    params={"company_id": COMPANY_ID}
)
projects = projects_response.json()
print(f"{len(projects)} projects found" if isinstance(projects, list) else "Failed to fetch projects")

StatementMeta(, b0f5b176-13c5-4283-b56f-2bd6dc11b560, 4, Finished, Available, Finished, False)

18 projects found


In [3]:
all_vendors = []

for project in projects:
    project_id = project["id"]
    project_name = project["name"]
    print(f"Pulling vendors for: {project_name}")

    page = 1
    while True:
        response = requests.get(
            f"https://api.procore.com/rest/v1.1/projects/{project_id}/vendors",
            headers=headers,
            params={
                "page": page,
                "per_page": 100
            }
        )

        if response.status_code != 200:
            print(f"  Error {response.status_code}, skipping")
            break

        rows = response.json()

        if not rows or isinstance(rows, dict):
            break

        for row in rows:
            row["project_id"] = project_id
            row["project_name"] = project_name

        all_vendors.extend(rows)

        if len(rows) < 100:
            break

        page += 1
        time.sleep(0.3)

print(f"Done! Total project vendors: {len(all_vendors)}")

StatementMeta(, b0f5b176-13c5-4283-b56f-2bd6dc11b560, 5, Finished, Available, Finished, False)

Pulling vendors for: 1100 Fulton Street
Pulling vendors for: 11 ESSEX ST
Pulling vendors for: 337A & 337B West Broadway Rehabilitaion Work
Pulling vendors for: 360 Lexington 8th & 20th Floor
Pulling vendors for: 549 Munroe Av
Pulling vendors for: 64 MET OVAL PSC + 1410 MET STOREROOM
Pulling vendors for: Boys & Girls Club
Pulling vendors for: EMBANKMENT PHASE II
Pulling vendors for: Embankment Phase III
Pulling vendors for: Embankment + Revetment Apartments 270 & 310 10th Street NJ
Pulling vendors for: Lillipvt 45 Renwick St
Pulling vendors for: PCNA 711 11TH AVE
Pulling vendors for: Sandbox Test Project
Pulling vendors for: SaunaLounge 45 South 3 Street, Brooklyn, NY
Pulling vendors for: Standard Project Template
Pulling vendors for: SYMRISE - 15th & 16th Flr
Pulling vendors for: TEST - ABM SUBORDINATE
Pulling vendors for: VOCO HOTEL TSQ
Done! Total project vendors: 384


In [4]:
import pandas as pd
import re

clean_rows = []
for row in all_vendors:
    clean_row = {}
    for key, value in row.items():
        if value is None:
            clean_row[key] = None
        elif isinstance(value, (dict, list)):
            clean_row[key] = json.dumps(value)
        elif isinstance(value, bool):
            clean_row[key] = str(value)
        elif isinstance(value, (int, float, str)):
            clean_row[key] = value
        else:
            clean_row[key] = str(value)
    clean_rows.append(clean_row)

def clean_column_name(col):
    col = col.strip()
    col = re.sub(r'[ ,;{}()\n\t=]', '_', col)
    col = re.sub(r'_+', '_', col)
    col = col.strip('_')
    return col

pdf = pd.DataFrame(clean_rows)
pdf.columns = [clean_column_name(c) for c in pdf.columns]

for col in pdf.columns:
    if pdf[col].dtype == object:
        pdf[col] = pdf[col].astype(str).replace('None', None)

spark.sql("DROP TABLE IF EXISTS procore_vendors_project_raw")

df = spark.createDataFrame(pdf)
df.write.format("delta").mode("append").saveAsTable("procore_project_vendors_raw")

print("Saved to Bronze_Lakehouse successfully")

StatementMeta(, b0f5b176-13c5-4283-b56f-2bd6dc11b560, 6, Finished, Available, Finished, False)

Saved to Bronze_Lakehouse successfully
